# 09_Feature_Recalculation
Recalculate all HSEP features for the technical skill universe.

In [32]:

import pandas as pd
import numpy as np
import re
from sklearn.preprocessing import MinMaxScaler
from pathlib import Path


## Load Datasets

In [33]:

tech = pd.read_csv('../Generated Datasets/technical_skill_master.csv')
history = pd.read_csv('../Generated Datasets/skill_demand_history_clean.csv')
global_adoption = pd.read_csv('../Generated Datasets/global_adoption_stackoverflow.csv')

linkedin_postings = pd.read_csv('../Raw Data/linkedin_job_postings.csv')
postings = pd.read_csv('../Raw Data/postings.csv')
job_skills = pd.read_csv('../Raw Data/job_skills.csv')

features = tech[['skill','sub_category','linkedin_frequency']].copy()

print(features.shape)


(114, 3)


## Feature 1: LinkedIn Demand

In [34]:

features['linkedin_demand'] = (
    features['linkedin_frequency'] /
    features['linkedin_frequency'].max()
)


## Feature 2: Salary Premium

In [35]:

salary_col = 'normalized_salary'
text_cols = [c for c in postings.columns if postings[c].dtype == 'object']
text_col = text_cols[0]

salary_scores = []

for skill in features['skill']:

    mask = postings[text_col].astype(str).str.contains(
        rf'\b{re.escape(skill)}\b',
        case=False,
        regex=True,
        na=False
    )

    subset = postings.loc[mask]

    if salary_col and len(subset) > 0:
        val = subset[salary_col].median()
    else:
        val = np.nan

    salary_scores.append(val)

features['salary_premium_raw'] = salary_scores

subcat_medians = (
    features.groupby('sub_category')['salary_premium_raw']
    .median()
    .to_dict()
)

features['salary_premium_raw'] = features.apply(
    lambda x: subcat_medians.get(x['sub_category'],0)
    if pd.isna(x['salary_premium_raw'])
    else x['salary_premium_raw'],
    axis=1
)

features['salary_premium'] = MinMaxScaler().fit_transform(
    features[['salary_premium_raw']]
)


c:\Users\Saanvi\anaconda3\envs\COURSE\Lib\site-packages\numpy\lib\_nanfunctions_impl.py:1213: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
c:\Users\Saanvi\anaconda3\envs\COURSE\Lib\site-packages\numpy\lib\_nanfunctions_impl.py:1213: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
c:\Users\Saanvi\anaconda3\envs\COURSE\Lib\site-packages\numpy\lib\_nanfunctions_impl.py:1213: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
c:\Users\Saanvi\anaconda3\envs\COURSE\Lib\site-packages\numpy\lib\_nanfunctions_impl.py:1213: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
c:\Users\Saanvi\anaconda3\envs\COURSE\Lib\site-packages\numpy\lib\_nanfunctions_impl.py:1213: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
c:\Users\Saanvi\anaconda3\envs\COURSE\Lib\site-packages\numpy\lib

## Feature 3: Current Usage

In [36]:

latest_year = history['year'].max()

latest = (
    history[history['year']==latest_year]
    [['skill','adoption_rate']]
    .rename(columns={'adoption_rate':'current_usage_raw'})
)

features = features.merge(latest,on='skill',how='left')


## Feature 4: Future Interest (StackOverflow 2025)

In [37]:

survey = pd.read_csv('../Raw Data/StackOverflow/survey_2025.csv')

want_cols = [
    c for c in survey.columns
    if 'WantToWorkWith' in c
]

interest_counts = {}

for col in want_cols:

    values = (
        survey[col]
        .dropna()
        .astype(str)
    )

    for row in values:

        for item in row.split(';'):
            item = item.strip().lower()

            interest_counts[item] = (
                interest_counts.get(item,0)+1
            )

features['future_interest_raw'] = (
    features['skill']
    .str.lower()
    .map(interest_counts)
)

features['future_interest_raw'] = (
    features['future_interest_raw']
    .fillna(0)
)


C:\Users\Saanvi\AppData\Local\Temp\ipykernel_79324\1241577141.py:1: DtypeWarning: Columns (56,74,92,97,98,105,109,110,132,162,165) have mixed types. Specify dtype option on import or set low_memory=False.
  survey = pd.read_csv('../Raw Data/StackOverflow/survey_2025.csv')


## Feature 5: Global Adoption

In [38]:

score_col = [
    c for c in global_adoption.columns
    if c.lower() != 'skill'
][0]

global_adoption = global_adoption.rename(
    columns={score_col:'global_adoption_score'}
)

features = features.merge(
    global_adoption[['skill','global_adoption_score']],
    on='skill',
    how='left'
)

features['global_adoption_score'] = (
    features['global_adoption_score']
    .fillna(features['global_adoption_score'].median())
)


## Feature 6: Growth Rate

In [39]:

growth_map = {}

for skill, grp in history.groupby('skill'):

    grp = grp.sort_values('year')

    if len(grp) < 2:
        continue

    years = grp['year'].values
    adoption = grp['adoption_rate'].values

    slope = np.polyfit(years, adoption, 1)[0]

    recent_growth = 0

    if len(grp) >= 3:
        recent = grp.tail(3)

        first = recent['adoption_rate'].iloc[0]
        last = recent['adoption_rate'].iloc[-1]

        if first != 0:
            recent_growth = (last-first)/first

    growth_map[skill] = (
        0.7*slope +
        0.3*recent_growth
    )

features['growth_rate_raw'] = (
    features['skill']
    .map(growth_map)
)

subcat_growth = (
    features.groupby('sub_category')['growth_rate_raw']
    .median()
    .to_dict()
)

features['growth_rate_raw'] = features.apply(
    lambda x: subcat_growth.get(x['sub_category'],0)
    if pd.isna(x['growth_rate_raw'])
    else x['growth_rate_raw'],
    axis=1
)


## Normalize Features

In [40]:

for col in [
    'current_usage_raw',
    'future_interest_raw',
    'growth_rate_raw'
]:
    features[col] = features[col].fillna(0)

features['current_usage'] = MinMaxScaler().fit_transform(
    features[['current_usage_raw']]
)

features['future_interest'] = MinMaxScaler().fit_transform(
    features[['future_interest_raw']]
)

features['growth_rate'] = MinMaxScaler().fit_transform(
    features[['growth_rate_raw']]
)


## Final Dataset

In [41]:

final_df = features[[
    'skill',
    'sub_category',
    'linkedin_demand',
    'salary_premium',
    'current_usage',
    'future_interest',
    'global_adoption_score',
    'growth_rate'
]]

print(final_df.shape)
final_df.head()


(114, 9)


,skill,sub_category,linkedin_demand,salary_premium,current_usage,future_interest,global_adoption_score,global_adoption_score,growth_rate
0,data analysis,Data Analytics,1.000000,NaN,0.000000,0.000000,160.0,160.000000,0.566533
1,excel,Data Analytics,0.506295,NaN,0.000000,0.000000,160.0,160.000000,0.566533
2,quality assurance,Software Engineering,0.442621,0.243541,0.000000,0.000000,160.0,160.000000,0.566533
3,microsoft excel,Data Analytics,0.314443,NaN,0.000000,0.000000,160.0,160.000000,0.566533
4,python,Programming,0.307062,0.000000,0.876458,0.701599,173.0,0.936412,0.255955


In [42]:
final_df = final_df.drop(columns=['global_adoption_score'])
final_df = final_df.rename(
    columns={'global_adoption_score.1':'global_adoption_score'}
)

In [43]:
final_df['salary_premium'] = (
    final_df.groupby('sub_category')['salary_premium']
      .transform(lambda x: x.fillna(x.median()))
)

final_df['salary_premium'] = (
    final_df['salary_premium']
      .fillna(final_df['salary_premium'].median())
)

c:\Users\Saanvi\anaconda3\envs\COURSE\Lib\site-packages\numpy\lib\_nanfunctions_impl.py:1213: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
c:\Users\Saanvi\anaconda3\envs\COURSE\Lib\site-packages\numpy\lib\_nanfunctions_impl.py:1213: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
c:\Users\Saanvi\anaconda3\envs\COURSE\Lib\site-packages\numpy\lib\_nanfunctions_impl.py:1213: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)


In [44]:
history_found = features['skill'].isin(history['skill']).astype(int)

In [45]:
final_df.to_csv(
    '../Generated Datasets/expanded_skill_master.csv',
    index=False
)

In [46]:
print(
    final_df.sort_values(
        'future_interest',
        ascending=False
    )[
        ['skill','future_interest']
    ].head(20)
)

               skill  future_interest
84            github         1.000000
4             python         0.701599
27            docker         0.691656
87        postgresql         0.670188
5                sql         0.635953
14        javascript         0.597763
55        typescript         0.570533
109           gitlab         0.431614
47             react         0.396814
69           node.js         0.384498
24        kubernetes         0.382577
23              jira         0.370714
39                c#         0.345574
25               c++         0.296198
68             mysql         0.289249
8               java         0.281397
92           mongodb         0.249760
58        confluence         0.240947
79   microsoft azure         0.231004
102     google cloud         0.225693


In [47]:
print(
    final_df.sort_values(
        'growth_rate',
        ascending=False
    )[
        ['skill','growth_rate']
    ].head(20)
)

                           skill  growth_rate
94                       pytorch     1.000000
13              machine learning     0.701560
38                  data science     0.701560
89                 deep learning     0.701560
98               computer vision     0.701560
42       artificial intelligence     0.701560
106  natural language processing     0.701560
86                     snowflake     0.647478
1                          excel     0.566533
2              quality assurance     0.566533
15                    automation     0.566533
16                data analytics     0.566533
17            data visualization     0.566533
3                microsoft excel     0.566533
12                       testing     0.566533
11          software development     0.566533
7         mechanical engineering     0.566533
6         electrical engineering     0.566533
36         business intelligence     0.566533
43                 cybersecurity     0.566533


In [48]:
important = [
    'langchain',
    'llm',
    'rag',
    'tableau',
    'power bi'
]

print(final_df[final_df['skill'].str.lower().isin(important)])

       skill    sub_category  linkedin_demand  salary_premium  current_usage  \
22   tableau  Data Analytics         0.085733         0.37969            0.0   
32  power bi  Data Analytics         0.066236         0.37969            0.0   

    future_interest  growth_rate  
22              0.0     0.566533  
32              0.0     0.566533  


In [49]:
print(final_df['sub_category'].value_counts())

sub_category
Software Engineering    20
Programming             18
Database                13
Data Analytics          12
DevOps                  12
AI_ML                    8
Engineering_Systems      7
Cloud                    7
Data Engineering         7
Cybersecurity            6
Tools_Platforms          4
Name: count, dtype: int64


In [51]:
print(final_df.columns.tolist())

['skill', 'sub_category', 'linkedin_demand', 'salary_premium', 'current_usage', 'future_interest', 'growth_rate']


In [50]:
new_skills = pd.DataFrame([
    {
        'skill': 'llm',
        'sub_category': 'AI_ML'
    },
    {
        'skill': 'langchain',
        'sub_category': 'AI_ML'
    },
    {
        'skill': 'rag',
        'sub_category': 'AI_ML'
    }
])

ai_reference = final_df[
    final_df['skill'].isin([
        'artificial intelligence',
        'natural language processing'
    ])
]

base = ai_reference.mean(numeric_only=True)

new_skills['linkedin_demand'] = base['linkedin_demand']
new_skills['salary_premium'] = base['salary_premium']
new_skills['current_usage'] = base['current_usage']
new_skills['future_interest'] = min(
    1.0,
    base['future_interest'] * 1.15
)
new_skills['global_adoption_score'] = base['global_adoption_score']
new_skills['growth_rate'] = min(
    1.0,
    base['growth_rate'] * 1.20
)

final_df = pd.concat(
    [final_df, new_skills],
    ignore_index=True
)

KeyError: 'global_adoption_score'